# Ljungqvist & Sargent (2018), Chapter 11 --- Fiscal Policies in a Growth Model

Companion notebook to `ls_solutions_ch11.pdf`. Chapter 11 has **20 end-of-chapter exercises** (printed pp. 449--470).
One section per exercise. Every number in the PDF is computed here and checked by an independent route with an `assert`:
Newton on the whole path vs the shooting algorithm of section 11.6.3 (bisection on $c_0$) vs the linear approximation
(11.10.8); the stable root $\lambda_2$ vs formula (11.10.16); value-function iteration vs closed forms; the household's
own problem at equilibrium prices vs the aggregate path (Big $K$, little $k$).

The LQ toolkit of `ls_ch05.ipynb`/`ls_ch07.ipynb` is not used: the models here are nonlinear growth models solved by
shooting, Newton and linearisation, which the shared tools below implement. Benchmark parameters (section 11.9) wherever
the book gives none: $\alpha=.33$, $\delta=.2$, $\beta=.95$, $\gamma=2$, $g=.2$, $f(k)=k^\alpha$; deviations are stated
in each section.

In [1]:
import logging
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq, linprog, minimize_scalar

plt.rcParams.update({"pdf.fonttype": 42, "font.size": 9})  # TrueType, never Type 3
FIG = "fig/"
logging.getLogger("fontTools").setLevel(logging.ERROR)  # silence font-subsetting timestamp notices
np.set_printoptions(precision=6, suppress=True, linewidth=110)

# Benchmark of section 11.9 (used wherever the book gives no numbers)
AL, DE, BE, GA, GB = .33, .2, .95, 2.0, .2
RHO = 1 / BE - 1
N = 150  # truncation date S of the shooting/Newton problems


def f(k):
    return k**AL


def fp(k):
    return AL * k**(AL - 1)


def fpp(k):
    return AL * (AL - 1) * k**(AL - 2)


def u(c, gam):
    return np.log(c) if gam == 1 else c**(1 - gam) / (1 - gam)


def kbar(tk=0.0):
    """Steady state (11.6.7): f'(k) = delta + rho/(1 - tau_k)."""
    return (AL / (DE + RHO / (1 - tk)))**(1 / (1 - AL))


def policy(n=N + 2, **kw):
    """Policy arrays t = 0..n-1 for g, tc, tk, psi (investment productivity, 11.3) and lev (levy per unit
    of k, 11.4). Scalars are constant paths; a short array is padded with its last value."""
    base = dict(g=GB, tc=0.0, tk=0.0, psi=1.0, lev=0.0)
    base.update(kw)
    out = {}
    for key, v in base.items():
        v = np.atleast_1d(np.asarray(v, float))
        out[key] = np.concatenate([v, np.full(max(n - len(v), 0), v[-1])])[:n]
    return out


def jump(before, after, T, n=N + 2):
    """Path equal to `before` for t < T and `after` for t >= T."""
    return np.where(np.arange(n) < T, before, after)


def cons(k, kn, p, t):
    """c_t from feasibility (11.6.1), with k_{t+1} = (1-delta)k_t + psi_t x_t."""
    return f(k) - p["g"][t] - (kn - (1 - DE) * k) / p["psi"][t]


def rbar(kn, p, t):
    """Rbar_{t+1} of (11.6.8e), generalised: psi_t[(1-tk)(f'-delta) + delta + (1-delta)/psi_{t+1} - lev]."""
    R = p["psi"][t] * ((1 - p["tk"][t + 1]) * (fp(kn) - DE) + DE + (1 - DE) / p["psi"][t + 1]
                       - p["lev"][t + 1])
    return (1 + p["tc"][t]) / (1 + p["tc"][t + 1]) * R


def kss(p):
    """Terminal steady state: beta * Rbar = 1 at the last policy values."""
    q = {key: v[-2:] for key, v in p.items()}
    return brentq(lambda k: BE * rbar(k, q, 0) - 1, 1e-8, 1e3)


def newton(resid, x, ncol=None, tol=1e-11, maxit=100):
    """Damped Newton with a finite-difference Jacobian. ncol (odd): if residual i depends only on unknowns
    i - ncol//2 .. i + ncol//2, columns i, i+ncol, ... are perturbed together (graph colouring)."""
    n = len(x)
    ncol = n if ncol is None else ncol
    F = resid(x)
    for _ in range(maxit):
        if np.max(np.abs(F)) < tol:
            return x
        J = np.zeros((len(F), n))
        h = 1e-7 * np.maximum(1.0, np.abs(x))
        for m in range(ncol):
            idx = np.arange(m, n, ncol)
            xp = x.copy()
            xp[idx] += h[idx]
            dF = (resid(xp) - F)[:, None]
            for i in idx:
                rows = slice(max(i - ncol // 2, 0), i + ncol // 2 + 1) if ncol < n else slice(None)
                J[rows, i] = dF[rows, 0] / h[i]
        dx = np.linalg.solve(J, -F)
        s = 1.0
        while s > 1e-6:
            with np.errstate(all="ignore"):
                Fn = resid(x + s * dx)
            if np.all(np.isfinite(Fn)) and np.max(np.abs(Fn)) < (1 - 1e-4 * s) * np.max(np.abs(F)):
                break
            s /= 2
        x, F = x + s * dx, Fn
    assert np.max(np.abs(F)) < tol, f"Newton failed: {np.max(np.abs(F))}"
    return x


def solve_path(k0, p, gam=GA, n=N, kT=None):
    """Equilibrium path by Newton (scipy hybr) on the Euler equations (11.6.3), t = 0..n-1,
    unknowns k_1..k_n, terminal condition k_{n+1} = kT. Returns k_0..k_{n+1}, c_0..c_n."""
    kT = kss(p) if kT is None else kT
    t = np.arange(n)

    def unpack(x):
        k = np.concatenate([[k0], x, [kT]])
        return k, cons(k[:-1], k[1:], p, np.arange(n + 1))

    def resid(x):
        k, c = unpack(x)
        return (c[1:] / c[:-1])**gam - BE * rbar(k[1:n + 1], p, t)

    x = newton(resid, kT + (k0 - kT) * 0.8**np.arange(1, n + 1), ncol=3)
    return unpack(x)


def shoot(k0, p, gam=GA, n=N, kT=None):
    """Shooting algorithm of section 11.6.3: bisection on c_0 until k_n ~ kT
    (c_0 too high -> k collapses; too low -> k overshoots). Returns k_0..k_n, c_0..c_n."""
    kT = kss(p) if kT is None else kT

    def run(c0):
        k, c = np.empty(n + 1), np.empty(n + 1)
        k[0], c[0] = k0, c0
        for t in range(n):
            k[t + 1] = (1 - DE) * k[t] + p["psi"][t] * (f(k[t]) - p["g"][t] - c[t])
            if k[t + 1] <= 0:
                return -1, k, c
            R = rbar(k[t + 1], p, t)
            if R <= 0 or k[t + 1] > 50 * kT:
                return 1, k, c
            c[t + 1] = c[t] * (BE * R)**(1 / gam)
        return np.sign(k[n] - kT), k, c

    lo, hi = 1e-10, f(k0) - p["g"][0] + (1 - DE) * k0 / p["psi"][0]
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if run(mid)[0] > 0:
            lo = mid
        else:
            hi = mid
        if hi - lo < 1e-16:
            break
    return run(lo)[1:]


def prices(k, c, p, gam=GA):
    """q_t (11.6.8b, normalised q_0 = 1), eta_t, w_t (11.6.8c-d), Rbar_{t+1} (11.6.8e), r_{t,t+1} = log(q_t/q_{t+1})."""
    n = len(c)
    t = np.arange(n)
    q = BE**t * c**(-gam) / (1 + p["tc"][t])
    q = q / q[0]
    return dict(q=q, eta=fp(k[:n]), w=f(k[:n]) - k[:n] * fp(k[:n]),
                Rb=rbar(k[1:n], p, t[:-1]), r=np.log(q[:-1] / q[1:]))


def pv(x, q):
    """sum_t q_t x_t with the tail after the last date at the terminal values (q_t ~ beta^t)."""
    return np.sum(q * x) + q[-1] * x[-1] * BE / (1 - BE)


def welfare(c, gam=GA):
    """sum_t beta^t u(c_t), tail at the last value of c."""
    t = np.arange(len(c))
    return np.sum(BE**t * u(c, gam)) + BE**len(c) * u(c[-1], gam) / (1 - BE)


def ce_gain(W1, W0, gam=GA):
    """Consumption-equivalent gain lam with W(c(1+lam)) = W1, W(c) = W0 (CRRA, gam != 1)."""
    return (W1 / W0)**(1 / (1 - gam)) - 1


def H(k0, k1, k2, z0, z1, gam):
    """H of (11.6.4): u'(c_t)/(1+tc_t) - beta u'(c_{t+1})/(1+tc_{t+1}) [(1-tk_{t+1})(f'(k_{t+1})-delta)+1];
    z = (g, tk, tc)."""
    c0 = f(k0) + (1 - DE) * k0 - z0[0] - k1
    c1 = f(k1) + (1 - DE) * k1 - z1[0] - k2
    return c0**(-gam) / (1 + z0[2]) - BE * c1**(-gam) / (1 + z1[2]) * ((1 - z1[1]) * (fp(k1) - DE) + 1)


def linear_path(k0, p, gam=GA, n=N):
    """Linear approximation (11.10.1)-(11.10.8) around the terminal steady state:
    k_{t+1} = lam2 k_t - (1/(phi0 lam1)) sum_j lam1^{-j} RHS_{t+j}. Returns k_0..k_{n+1}, lam1, lam2."""
    zb = np.array([p["g"][-1], p["tk"][-1], p["tc"][-1]])
    Z = np.column_stack([p["g"], p["tk"], p["tc"]]) - zb
    kb = kss(p)
    e = 1e-6
    x = [kb, kb, kb, zb, zb]
    d = []
    for i in range(3):
        xp, xm = list(x), list(x)
        xp[i] = kb + e
        xm[i] = kb - e
        d.append((H(*xp, gam) - H(*xm, gam)) / (2 * e))
    phi2, phi1, phi0 = d
    Hz = np.zeros((2, 3))
    for j in range(2):
        for i in range(3):
            zp, zm = zb.copy(), zb.copy()
            zp[i] += e
            zm[i] -= e
            xp, xm = list(x), list(x)
            xp[3 + j], xm[3 + j] = zp, zm
            Hz[j, i] = (H(*xp, gam) - H(*xm, gam)) / (2 * e)
    lam = np.sort(np.roots([phi0, phi1, phi2]).real)
    lam2, lam1 = lam
    rhs = -(Z[:-1] @ Hz[0] + Z[1:] @ Hz[1])
    F = np.zeros(len(rhs) + 1)
    for t in range(len(rhs) - 1, -1, -1):
        F[t] = rhs[t] + F[t + 1] / lam1
    kh = np.empty(n + 2)
    kh[0] = k0 - kb
    for t in range(n + 1):
        kh[t + 1] = lam2 * kh[t] - F[t] / (phi0 * lam1)
    return kb + kh, lam1, lam2


def lam2_root(gam, g=GB):
    """Stable root of lambda^2 - b lambda + 1/beta = 0, b = 1 + 1/beta + beta c |f''(k)|/gam (no distortions)."""
    k = kbar()
    c = f(k) - DE * k - g
    b = 1 + 1 / BE + BE * c * abs(fpp(k)) / gam
    return (b - np.sqrt(b * b - 4 / BE)) / 2

## Exercise 11.1 --- Tax reform: I

(d) A constant $\tau_c=\bar g/\bar c$ from $t=0$ leaves the allocation at the old steady state (it cancels from (11.6.3)).
(e) $\tau_c=0$ for $t\le9$, $\bar\tau_c$ for $t\ge10$: $\bar\tau_c$ solves the present-value budget (1) at the
equilibrium prices (fixed point by `brentq`). The foreseen rise acts like a one-time capital tax between $t=9$ and $10$.
(f) Welfare of both. Checks: Newton vs shooting; allocation of (d) equals the old steady state.

In [2]:
# Exercise 11.1: tau_c from t = 0 (d) vs tau_c postponed to t = 10 (e); lump sums ruled out.
k0 = kbar()
cbar = f(k0) - DE * k0 - GB
tau_d = GB / cbar                                  # (d): constant tau_c balancing (1)
pd_ = policy(tc=tau_d)
kd, cd = solve_path(k0, pd_)
assert np.allclose(kd, k0) and np.allclose(cd, cbar)   # constant tau_c: allocation unchanged
qd = prices(kd, cd, pd_)["q"]
assert np.isclose(pv(tau_d * cd, qd), pv(GB + 0 * cd, qd))


def gap11_1(tau):
    """PV of consumption-tax revenue minus PV of g when tau_c = 0 (t < 10), tau (t >= 10)."""
    p = policy(tc=jump(0.0, tau, 10))
    k, c = solve_path(k0, p)
    q = prices(k, c, p)["q"]
    return pv(np.where(np.arange(len(c)) >= 10, tau * c, 0.0), q) - pv(GB + 0 * c, q)


tau_e = brentq(gap11_1, 0.3, 1.2)
pe = policy(tc=jump(0.0, tau_e, 10))
ke, ce = solve_path(k0, pe)
ks, cs = shoot(k0, pe)
assert np.allclose(ks[:40], ke[:40], atol=1e-8) and np.allclose(cs[:40], ce[:40], atol=1e-8)
Wd, We = welfare(cd), welfare(ce)
assert Wd > We and np.isclose(kss(pe), k0)
Rb = prices(ke, ce, pe)["Rb"]
print(f"k0 = {k0:.6f}, cbar = {cbar:.6f}; (d) tau_c = {tau_d:.6f}; (e) tau_c-bar = {tau_e:.6f}")
print(f"(e) c_0 = {ce[0]:.6f}, c_9 = {ce[9]:.6f}, c_10 = {ce[10]:.6f}, k_10 = {ke[10]:.6f}, min k = {ke.min():.6f}; "
      f"Rbar_10 = {Rb[9]:.6f}")
print(f"welfare (d) {Wd:.6f} > (e) {We:.6f}; consumption-equivalent loss of (e) = {-ce_gain(We, Wd):.4%}")
fig, ax = plt.subplots(1, 3, figsize=(9, 2.4))
t = np.arange(41)
ax[0].plot(t, ke[:41], "k-", t, kd[:41], "k--"); ax[0].set_title("$k_t$")
ax[1].plot(t, ce[:41], "k-", t, cd[:41], "k--"); ax[1].set_title("$c_t$")
ax[2].plot(t[1:], Rb[:40], "k-", t, 0 * t + 1 / BE, "k--"); ax[2].set_title(r"$\bar R_{t}$")
fig.tight_layout(); fig.savefig(FIG + "ls11_01_tau_c.pdf"); plt.close(fig)

k0 = 1.489956, cbar = 0.642645; (d) tau_c = 0.311214; (e) tau_c-bar = 0.623899
(e) c_0 = 0.658407, c_9 = 0.702440, c_10 = 0.564675, k_10 = 1.126850, min k = 1.126850; Rbar_10 = 0.680229
welfare (d) -31.121369 > (e) -31.257355; consumption-equivalent loss of (e) = 0.4351%


## Exercise 11.2 --- Tax reform: II

Steady-state capital-tax revenue $\tau_k\rho k/(1-\tau_k)$ peaks below $g=.2$, so we use $g=.05$. (b) $\tau_k$ on the
low side of the Laffer curve. (c) $\tau_k=0$ and a constant $\tau_c$: planner's path from $\bar k_0$, $\tau_c$ from the
PV budget. (d) Steady-state welfare comparison vs the comparison that includes the transition.

In [3]:
# Exercise 11.2: g financed by a constant tau_k (b), then by a constant tau_c (c); welfare (d).
def rev_k(tk):
    """Steady-state capital-tax revenue tau_k (f'(k) - delta) k = tau_k rho k / (1 - tau_k)."""
    k = kbar(tk)
    return tk * (fp(k) - DE) * k


G2 = 0.05  # g = .2 exceeds the maximum steady-state capital-tax revenue; see the print below
tgrid = np.linspace(0, 0.99, 991)
rgrid = np.array([rev_k(x) for x in tgrid])
tk_peak = tgrid[rgrid.argmax()]
tk0 = brentq(lambda x: rev_k(x) - G2, 1e-6, tk_peak)            # low-tax side of the Laffer curve
k0 = kbar(tk0)
c0 = f(k0) - DE * k0 - G2
assert np.isclose(tk0 * RHO * k0 / (1 - tk0), G2) and np.isclose(fp(k0), DE + RHO / (1 - tk0))
p = policy(g=G2)                                                 # tau_k = 0; constant tau_c cancels in (11.6.3)
k, c = solve_path(k0, p)
ks, cs = shoot(k0, p)
assert np.allclose(ks[:40], k[:40], atol=1e-8)
q = prices(k, c, p)["q"]
tc_new = pv(G2 + 0 * c, q) / pv(c, q)
k1 = kbar()
c1 = f(k1) - DE * k1 - G2
assert np.isclose(k[-1], k1) and c1 > c0 and np.all(np.diff(k) > 0)
W_ss0, W_ss1, W_path = u(c0, GA) / (1 - BE), u(c1, GA) / (1 - BE), welfare(c)
print(f"max steady-state capital-tax revenue {rgrid.max():.4f} < g = {GB} of section 11.9; we use g = {G2}")
print(f"(b) tau_k = {tk0:.6f} (Laffer peak at tau_k ~ {tk_peak:.2f}, max revenue {rgrid.max():.4f}); "
      f"k = {k0:.6f}, c = {c0:.6f}, pre-tax f'(k)+1-delta = {fp(k0) + 1 - DE:.6f}, Rbar = {1 + RHO:.6f}")
print(f"(c) new steady state k = {k1:.6f}, c = {c1:.6f}, f'(k)+1-delta = {fp(k1) + 1 - DE:.6f}; "
      f"constant tau_c = {tc_new:.6f} (steady-state g/c = {G2 / c1:.6f}); c_0 = {c[0]:.6f}")
print(f"(d) steady-state comparison: gain {ce_gain(W_ss1, W_ss0):.4%}; with the transition from k0: "
      f"gain {ce_gain(W_path, W_ss0):.4%}")
assert ce_gain(W_ss1, W_ss0) > ce_gain(W_path, W_ss0) > 0 and c[0] < c0
fig, ax = plt.subplots(1, 3, figsize=(9, 2.4))
t = np.arange(41)
ax[0].plot(t, k[:41], "k-", t, 0 * t + k0, "k--"); ax[0].set_title("$k_t$")
ax[1].plot(t, c[:41], "k-", t, 0 * t + c0, "k--"); ax[1].set_title("$c_t$")
ax[2].plot(tgrid, rgrid, "k-", tgrid, 0 * tgrid + G2, "k:"); ax[2].set_title(r"steady-state revenue vs $\tau_k$")
fig.tight_layout(); fig.savefig(FIG + "ls11_02_tax_reform.pdf"); plt.close(fig)

max steady-state capital-tax revenue 0.1461 < g = 0.2 of section 11.9; we use g = 0.05
(b) tau_k = 0.445443 (Laffer peak at tau_k ~ 0.91, max revenue 0.1461); k = 1.182709, c = 0.770397, pre-tax f'(k)+1-delta = 1.094907, Rbar = 1.052632
(c) new steady state k = 1.489956, c = 0.792645, f'(k)+1-delta = 1.052632; constant tau_c = 0.064569 (steady-state g/c = 0.063080); c_0 = 0.721969
(d) steady-state comparison: gain 2.8879%; with the transition from k0: gain 0.5953%


## Exercise 11.3 --- Anticipated productivity shift

Investment-specific productivity $\psi_t$ (1 for $t<4$, 2 for $t\ge4$), no government ($g=0$).
Euler: $u'(c_t)/\psi_t=\beta u'(c_{t+1})[f'(k_{t+1})+(1-\delta)/\psi_{t+1}]$. New steady state
$f'(k)=(\rho+\delta)/\psi$. Checks: Newton vs shooting; steady-state formula.

In [4]:
# Exercise 11.3: investment-specific productivity psi_t = 1 (t < 4), 2 (t >= 4); no government.
p = policy(g=0.0, psi=jump(1.0, 2.0, 4))
k0 = kbar()                                    # old steady state f'(k) = rho + delta (psi = 1)
kT = kss(p)
assert np.isclose(fp(kT), (RHO + DE) / 2) and np.isclose(kT, 2**(1 / (1 - AL)) * k0)
k, c = solve_path(k0, p)
ks, cs = shoot(k0, p)
assert np.allclose(ks[:40], k[:40], atol=1e-8) and np.allclose(cs[:40], c[:40], atol=1e-8)
t = np.arange(len(c))
x = (k[1:] - (1 - DE) * k[:-1]) / p["psi"][t]
lam = c**(-GA) / p["psi"][t]                   # price of installed capital q_t / psi_t (normalised)
# steady-state comparative statics (a): k ~ psi^{1/(1-alpha)}, x = delta k / psi, c = f(k) - delta k / psi
for ps in (1.0, 2.0):
    kk = ((RHO + DE) / (AL * ps))**(1 / (AL - 1))
    print(f"psi = {ps}: k = {kk:.6f}, x = {DE * kk / ps:.6f}, c = {f(kk) - DE * kk / ps:.6f}")
print("t   k_t        c_t       x_t       gross return psi_t[f'(k_t+1)+(1-delta)/psi_t+1]")
for s in range(8):
    print(f"{s}  {k[s]:.6f}  {c[s]:.6f}  {x[s]:.6f}  {rbar(k[s + 1], p, s):.6f}")
c_old = f(k0) - DE * k0
assert c[0] > c_old and x[3] < DE * k0 and x[4] > x[3]       # consumption rises and investment dips before t = 4
fig, ax = plt.subplots(1, 3, figsize=(9, 2.4))
tt = np.arange(31)
ax[0].plot(tt, k[:31], "k-", tt, 0 * tt + k0, "k--"); ax[0].set_title("$k_t$")
ax[1].plot(tt, c[:31], "k-", tt, 0 * tt + c_old, "k--"); ax[1].set_title("$c_t$")
ax[2].plot(tt, x[:31], "k-", tt, 0 * tt + DE * k0, "k--"); ax[2].set_title("$x_t$")
fig.tight_layout(); fig.savefig(FIG + "ls11_03_psi.pdf"); plt.close(fig)

psi = 1.0: k = 1.489956, x = 0.297991, c = 0.842645
psi = 2.0: k = 4.192491, x = 0.419249, c = 1.185532
t   k_t        c_t       x_t       gross return psi_t[f'(k_t+1)+(1-delta)/psi_t+1]
0  1.489956  0.900940  0.239696  1.059478
1  1.431661  0.903866  0.221846  1.067616
2  1.367175  0.910276  0.198444  1.077924
3  1.292184  0.921147  0.167124  0.691912
4  1.200871  0.746821  0.315447  1.283411
5  1.591590  0.824634  0.341113  1.221116
6  1.955499  0.888180  0.359533  1.179564
7  2.283466  0.940208  0.373008  1.150407


## Exercise 11.4 --- A capital levy

Bill $\phi k_T$ in time-$T$ goods, $A=1$. $T=0$: lump sum, $\phi=\bar g/((1-\beta)\bar k_0)$. $T=10$ foreseen: the
return on $k_{10}$ becomes $f'(k_{10})+1-\delta-\phi$, and the levy has a Laffer curve: an equilibrium exists only for
small $\bar g$ (computed). We use $\bar g=.02$. (e) Government debt $B_t$ (11.4.3), checked against the sequential
budget (11.4.2).

In [5]:
# Exercise 11.4: capital levy phi on k_T (bill phi k_T in time-T goods), T = 0 or T = 10; A = 1.
k0 = kbar()                                  # (b) g = 0 steady state: alpha A k^(alpha-1) = rho + delta
TL = 10


def levy_eq(phi, g, T=TL):
    """Equilibrium path with the levy at T; returns k, c, prices, revenue/PV(g)."""
    p = policy(g=g, lev=np.where(np.arange(N + 2) == T, phi, 0.0))
    k, c = solve_path(k0, p)
    pr = prices(k, c, p)
    return k, c, pr, phi * pr["q"][T] * k[T] / pv(g + 0 * c, pr["q"])


def max_ratio(g):
    """Largest revenue/PV(g) the T = 10 levy can raise (peak of the Laffer curve in phi)."""
    r = minimize_scalar(lambda ph: -levy_eq(ph, g)[3], bounds=(0.3, 12.0), method="bounded", options={"xatol": 1e-6})
    return -r.fun, r.x


g_max = brentq(lambda g: max_ratio(g)[0] - 1, 0.01, 0.2, xtol=1e-8)
print(f"T = 10 levy: an equilibrium exists iff g <= {g_max:.5f}; e.g. g = .2 raises at most "
      f"{max_ratio(0.2)[0]:.4f} of PV(g) (peak phi = {max_ratio(0.2)[1]:.3f})")
G4 = 0.02
# (c) T = 0: k_0 is predetermined, the levy is lump sum; k stays at k0, c drops by g
p0 = policy(g=G4)
kc, cc = solve_path(k0, p0)
assert np.allclose(kc, k0) and np.allclose(cc, f(k0) - DE * k0 - G4)
phi0 = G4 / ((1 - BE) * k0)
assert np.isclose(phi0 * k0, pv(G4 + 0 * cc, prices(kc, cc, p0)["q"]))
# (d) T = 10: phi on the good side of the Laffer curve
phi_pk = max_ratio(G4)[1]
phi10 = brentq(lambda ph: levy_eq(ph, G4)[3] - 1, 1e-4, phi_pk)
kd, cd, prd, _ = levy_eq(phi10, G4)
pd_ = policy(g=G4, lev=np.where(np.arange(N + 2) == TL, phi10, 0.0))
ks, cs = shoot(k0, pd_)
assert np.allclose(ks[:40], kd[:40], atol=1e-8) and np.isclose(kd[-1], k0)
print(f"(c) T = 0: phi = g/((1-beta) k0) = {phi0:.6f}; k = {k0:.6f}, c = {cc[0]:.6f} (was {f(k0) - DE * k0:.6f}), r = {-np.log(BE):.6f}")
print(f"(d) T = 10, g = {G4}: phi = {phi10:.6f} (Laffer peak phi = {phi_pk:.3f}); k_10 = {kd[10]:.6f}; "
      f"c_0 = {cd[0]:.6f}, c_9 = {cd[9]:.6f}, c_10 = {cd[10]:.6f}; r_9,10 = {prd['r'][9]:.6f}, r_10,11 = {prd['r'][10]:.6f}")
assert kd[10] < k0 and cd[0] > cc[0] and prd["r"][9] < -np.log(BE) < prd["r"][10]
Wc, Wd = welfare(cc), welfare(cd)
print(f"welfare T = 0: {Wc:.6f} > T = 10: {Wd:.6f}")
assert Wc > Wd
# (e) government debt B_t = sum_{s>t} q_s/q_t (T_s - g_s)  (11.4.3); household bonds = B_t
q = prd["q"]
n = len(q)
Tax = np.where(np.arange(n) == TL, phi10 * kd[TL], 0.0)
surplus = Tax - G4
B = np.array([(np.sum(q[s + 1:] * surplus[s + 1:]) - q[-1] * G4 * BE / (1 - BE)) / q[s] for s in range(n)])
# sequential budget (11.4.2): g_t + R_{t-1,t} B_{t-1} = T_t + B_t
Rg = q[:-1] / q[1:]
assert np.allclose(G4 + Rg[:40] * B[:40], Tax[1:41] + B[1:41], atol=1e-9)
assert np.all(B[:TL] > 0) and np.all(B[TL:60] < 0) and np.isclose(B[-1], -G4 / RHO, rtol=1e-3)
print(f"(e) B_0 = {B[0]:.6f}, B_9 = {B[9]:.6f}, B_10 = {B[10]:.6f}, B_inf = -g/rho = {-G4 / RHO:.6f}")
fig, ax = plt.subplots(1, 4, figsize=(11, 2.4))
t = np.arange(41)
ax[0].plot(t, kd[:41], "k-", t, kc[:41], "k--"); ax[0].set_title("$k_t$")
ax[1].plot(t, cd[:41], "k-", t, cc[:41], "k--", t, 0 * t + f(k0) - DE * k0, "k:"); ax[1].set_title("$c_t$")
ax[2].plot(t[:40], prd["r"][:40], "k-", t, 0 * t - np.log(BE), "k--"); ax[2].set_title("$r_{t,t+1}$")
ax[3].plot(t, B[:41], "k-"); ax[3].axhline(0, color="gray", lw=.5); ax[3].set_title("$B_t$ (gov. debt = household bonds)")
fig.tight_layout(); fig.savefig(FIG + "ls11_04_levy.pdf"); plt.close(fig)

T = 10 levy: an equilibrium exists iff g <= 0.03541; e.g. g = .2 raises at most 0.3228 of PV(g) (peak phi = 5.799)


(c) T = 0: phi = g/((1-beta) k0) = 0.268464; k = 1.489956, c = 0.822645 (was 0.842645), r = 0.051293
(d) T = 10, g = 0.02: phi = 0.494143 (Laffer peak phi = 1.516); k_10 = 1.012095; c_0 = 0.840408, c_9 = 0.911460, c_10 = 0.706924; r_9,10 = -0.456955, r_10,11 = 0.106263
welfare T = 0: -24.311816 > T = 10: -24.441775
(e) B_0 = 0.020000, B_9 = 0.287525, B_10 = -0.298057, B_inf = -g/rho = -0.380000


## Exercise 11.5 --- Log utility, golden rule, unforeseen jump in $g$

Checks: modified golden rule below the golden rule; after $g=\bar c/2$ at $t=0$ the Newton path is flat ($k$ and
$\bar R$ constant, $c$ halves).

In [6]:
# Exercise 11.5: log utility; modified vs golden rule; unforeseen permanent jump g = cbar/2 at t = 0.
k5 = (AL / (RHO + DE))**(1 / (1 - AL))       # (a) with z = 1
kg = (AL / DE)**(1 / (1 - AL))               # (b) golden rule f'(k) = delta
c5, cg = f(k5) - DE * k5, f(kg) - DE * kg
assert np.isclose(k5, kbar()) and kg > k5 and cg > c5
p = policy(g=c5 / 2)
k, c = solve_path(k5, p, gam=1.0)
assert np.allclose(k, k5) and np.allclose(c, c5 / 2)
Rb = prices(k, c, p, gam=1.0)["Rb"]
assert np.allclose(Rb, 1 + RHO)
print(f"(a) kbar = {k5:.6f}, cbar = {c5:.6f}; (b) golden rule k = {kg:.6f} > kbar, c = {cg:.6f} > cbar")
print(f"(c) g = cbar/2 = {c5 / 2:.6f}: k_t = {k[0]:.6f}, c_t = {c[0]:.6f}, Rbar = {Rb[0]:.6f} for all t")

(a) kbar = 1.489956, cbar = 0.842645; (b) golden rule k = 2.111557 > kbar, c = 0.857420 > cbar
(c) g = cbar/2 = 0.421323: k_t = 1.489956, c_t = 0.421323, Rbar = 1.052632 for all t


## Exercise 11.6 --- Trade and growth

Closed economy from $\bar k_0=.5\bar k$ vs small open economy with $R=\beta^{-1}$: $k_t=\bar k$, $B_t=B_{-1}=\bar k-\bar k_0$,
$c=f(\bar k)-\delta\bar k-\rho B_{-1}$. Checks: PV and period budgets; welfare ordering.

In [7]:
# Exercise 11.6: closed economy from kbar_0 = .5 kbar vs small open economy with R = 1/beta.
kb = kbar()
kb0 = 0.5 * kb
p = policy(g=0.0)
k, c = solve_path(kb0, p)
Rb = prices(k, c, p)["Rb"]
assert np.all(np.diff(k) > 0) and np.all(np.diff(c) > 0) and np.all(Rb > 1 + RHO) and np.all(np.diff(Rb) < 0)
# open economy: k_t = kbar for t >= 0, B_t = B_{-1} = kbar - kbar_0, c = f(kbar) - delta kbar - rho B_{-1}
Bm1 = kb - kb0
c_open = f(kb) - DE * kb - RHO * Bm1
# check the budget in present value: sum beta^t c = sum beta^t (f - delta kbar) - R B_{-1}
assert np.isclose(c_open / (1 - BE), (f(kb) - DE * kb) / (1 - BE) - Bm1 / BE)
# check the period budget with constant debt: c + k + R B = f(k) + (1-delta) k + B
assert np.isclose(c_open + kb + Bm1 / BE, f(kb) + (1 - DE) * kb + Bm1)
W_closed, W_open = welfare(c), u(c_open, GA) / (1 - BE)
assert W_open > W_closed
print(f"closed: k_0 = {kb0:.6f} -> {kb:.6f}; c_0 = {c[0]:.6f} -> {c[-1]:.6f}; Rbar_1 = {Rb[0]:.6f} -> {1 + RHO:.6f}")
print(f"open: B_-1 = {Bm1:.6f}, k_t = {kb:.6f}, c_t = {c_open:.6f}, interest paid rho B = {RHO * Bm1:.6f}")
print(f"welfare closed {W_closed:.6f} < open {W_open:.6f}; consumption-equivalent gain {ce_gain(W_open, W_closed):.4%}")
fig, ax = plt.subplots(1, 3, figsize=(9, 2.4))
t = np.arange(41)
ax[0].plot(t, k[:41], "k-", t, 0 * t + kb, "k--"); ax[0].set_title("$k_t$ (dashed: open)")
ax[1].plot(t, c[:41], "k-", t, 0 * t + c_open, "k--"); ax[1].set_title("$c_t$ (dashed: open)")
ax[2].plot(t[1:], Rb[:40], "k-", t, 0 * t + 1 + RHO, "k--"); ax[2].set_title(r"$\bar R_{t}$")
fig.tight_layout(); fig.savefig(FIG + "ls11_06_open.pdf"); plt.close(fig)

closed: k_0 = 0.744978 -> 1.489956; c_0 = 0.647540 -> 0.842645; Rbar_1 = 1.166272 -> 1.052632
open: B_-1 = 0.744978, k_t = 1.489956, c_t = 0.803436, interest paid rho B = 0.039209
welfare closed -25.239350 < open -24.893088; consumption-equivalent gain 1.3910%


## Exercise 11.7 --- Capital tax, unforeseen (b) and foreseen (d); role of $\gamma$ (c)

$\tau_k=.2$; $\gamma\in\{.2,2,5\}$. Checks: Newton vs shooting; linear approximation (11.10.8) vs Newton, with an error
that shrinks in proportion to the size of the tax change; $\lambda_2$ from the characteristic polynomial vs (11.10.16):
the formula as printed is wrong, the corrected one matches.

In [8]:
# Exercise 11.7: permanent tau_k = .2, unforeseen at t = 0 (b, c) or announced at 0 for t = 10 (d).
TK = 0.2
kb, kn = kbar(), kbar(TK)
cb, cn = f(kb) - DE * kb - GB, f(kn) - DE * kn - GB
print(f"(a) k = {kb:.6f}, c = {cb:.6f}, Rbar = {1 + RHO:.6f};  new: k = {kn:.6f}, c = {cn:.6f}, Rbar = {1 + RHO:.6f}")
res7 = {}
for gam in (0.2, 2.0, 5.0):
    for name, T in (("b", 0), ("d", 10)):
        p = policy(tk=jump(0.0, TK, T))
        k, c = solve_path(kb, p, gam=gam)
        ks, cs = shoot(kb, p, gam=gam)
        assert np.allclose(ks[:30], k[:30], atol=1e-7) and np.allclose(cs[:30], c[:30], atol=1e-7)
        err = [np.max(np.abs(k - linear_path(kb, p, gam=gam)[0])) / np.max(np.abs(k - kn))]
        for tk in (0.02, 0.002):                     # the linear error is first order in the size of the change
            pp = policy(tk=jump(0.0, tk, T))
            kk = solve_path(kb, pp, gam=gam)[0]
            err.append(np.max(np.abs(kk - linear_path(kb, pp, gam=gam)[0])) / np.max(np.abs(kk - kbar(tk))))
        assert err[2] < 2e-3 and 5 < err[1] / err[2] < 15
        print(f"gamma = {gam}, T = {T}: linear (11.10.8) vs Newton, max error / max deviation for tau_k = .2, .02, .002:",
              np.round(err, 5))
        res7[name, gam] = k, c, prices(k, c, p, gam)["Rb"]
for gam in (0.2, 2.0, 5.0):
    k, c, Rb = res7["b", gam]
    assert np.all(np.diff(k[:40]) < 0) and np.all(np.diff(c[:40]) < 0) and c[0] > cb and c[-1] < cb and np.all(Rb < 1 + RHO + 1e-12)
    kd, cd, Rd = res7["d", gam]
    assert np.all(np.diff(kd[:11]) < 0) and np.all(np.diff(cd[:10]) > 0) and Rd[9] < Rd[8]
    print(f"gamma = {gam}: (b) c_0 = {c[0]:.6f}, k_5 = {k[5]:.6f}, Rbar_1 = {Rb[0]:.6f}; "
          f"(d) c_0 = {cd[0]:.6f}, c_9 = {cd[9]:.6f}, c_10 = {cd[10]:.6f}, k_10 = {kd[10]:.6f}, "
          f"Rbar_9 = {Rd[8]:.6f}, Rbar_10 = {Rd[9]:.6f}; half-life of k-gap after T: "
          f"{np.argmax(np.abs(kd[10:] - kn) < 0.5 * abs(kd[10] - kn))}")
# (c) initial jump in c smaller and adjustment slower for larger gamma
assert res7["b", 0.2][1][0] > res7["b", 2.0][1][0] > res7["b", 5.0][1][0]
# (11.10.16): stable root lambda_2(gamma), as printed and corrected, vs the root of the characteristic polynomial
a = [.975, .0329, .0642, .00063, .0011]
printed = lambda g: g / (a[0] / g + a[1] + a[2] * np.sqrt(1 / g + a[3] / g**2 + a[4]))
fixed = lambda g: g / (a[0] * g + a[1] + np.sqrt(a[2] * g + a[3] * g**2 + a[4]))
for gam in (0.2, 1.0, 2.0, 5.0):
    lin = linear_path(kb, policy(), gam=gam)[2]
    assert np.isclose(lin, lam2_root(gam), atol=1e-6) and np.isclose(fixed(gam), lam2_root(gam), atol=2e-3)
    print(f"gamma = {gam}: lambda_2 root {lam2_root(gam):.5f}, numerical H {lin:.5f}, corrected (11.10.16) "
          f"{fixed(gam):.5f}, as printed {printed(gam):.5f}")
k_ = kbar(); c_ = f(k_) - DE * k_ - GB; A_ = BE * c_ * abs(fpp(k_))
assert np.allclose([(1 + BE) / 2, BE * A_ / 2, BE**2 * (1 + 1 / BE) * A_ / 2, ((1 - BE) / 2)**2, (BE * A_ / 2)**2],
                   a, atol=6e-5)
fig, ax = plt.subplots(2, 3, figsize=(9, 4.6))
t = np.arange(41)
for row, name in enumerate(("b", "d")):
    for gam, st in ((0.2, "k-."), (2.0, "k-"), (5.0, "k:")):
        k, c, Rb = res7[name, gam]
        ax[row, 0].plot(t, k[:41], st, lw=1.1); ax[row, 1].plot(t, c[:41], st, lw=1.1, label=rf"$\gamma={gam}$")
        ax[row, 2].plot(t[1:], Rb[:40], st, lw=1.1)
    for j, (y, ti) in enumerate(((kb, "$k_t$"), (cb, "$c_t$"), (1 + RHO, r"$\bar R_t$"))):
        ax[row, j].axhline(y, color="gray", ls="--", lw=.7)
        ax[row, j].set_title(ti + (" (unforeseen, t = 0)" if name == "b" else " (foreseen, t = 10)"), fontsize=8)
ax[0, 1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG + "ls11_07_tau_k.pdf"); plt.close(fig)
fig, ax = plt.subplots(figsize=(4, 2.6))
gg = np.linspace(0.05, 7, 200)
ax.plot(gg, [lam2_root(x) for x in gg], "k-", label="root of characteristic polynomial")
ax.plot(gg, fixed(gg), "k--", label="(11.10.16) corrected")
ax.set_xlabel(r"$\gamma$"); ax.set_ylabel(r"$\lambda_2$"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG + "ls11_07_lambda2.pdf"); plt.close(fig)

(a) k = 1.489956, c = 0.642645, Rbar = 1.052632;  new: k = 1.381220, c = 0.636222, Rbar = 1.052632
gamma = 0.2, T = 0: linear (11.10.8) vs Newton, max error / max deviation for tau_k = .2, .02, .002: [0.00651 0.0006  0.00006]
gamma = 0.2, T = 10: linear (11.10.8) vs Newton, max error / max deviation for tau_k = .2, .02, .002: [0.14902 0.01308 0.00129]
gamma = 2.0, T = 0: linear (11.10.8) vs Newton, max error / max deviation for tau_k = .2, .02, .002: [0.00515 0.00042 0.00004]


gamma = 2.0, T = 10: linear (11.10.8) vs Newton, max error / max deviation for tau_k = .2, .02, .002: [0.06919 0.00611 0.0006 ]
gamma = 5.0, T = 0: linear (11.10.8) vs Newton, max error / max deviation for tau_k = .2, .02, .002: [0.00745 0.00058 0.00006]
gamma = 5.0, T = 10: linear (11.10.8) vs Newton, max error / max deviation for tau_k = .2, .02, .002: [0.04695 0.00414 0.00041]
gamma = 0.2: (b) c_0 = 0.686913, k_5 = 1.389424, Rbar_1 = 1.046231; (d) c_0 = 0.642841, c_9 = 0.670493, c_10 = 0.656614, k_10 = 1.424950, Rbar_9 = 1.056715, Rbar_10 = 1.048237; half-life of k-gap after T: 2
gamma = 2.0: (b) c_0 = 0.657543, k_5 = 1.433437, Rbar_1 = 1.043471; (d) c_0 = 0.644886, c_9 = 0.650185, c_10 = 0.648307, k_10 = 1.442275, Rbar_9 = 1.057049, Rbar_10 = 1.046558; half-life of k-gap after T: 5
gamma = 5.0: (b) c_0 = 0.651494, k_5 = 1.452476, Rbar_1 = 1.042913; (d) c_0 = 0.645074, c_9 = 0.647047, c_10 = 0.646190, k_10 = 1.451417, Rbar_9 = 1.056377, Rbar_10 = 1.045685; half-life of k-gap after T

## Exercise 11.8 --- Trade and growth, version II

$\bar k_0=2\bar k$. Closed path; sell-all plan $c=(1-\beta)\bar k_0$ (constraint as printed); optimal open plan
$c=f(\bar k)-\delta\bar k+\rho(\bar k_0-\bar k)$. Threshold $\bar k_0$ above which selling everything beats autarky.

In [9]:
# Exercise 11.8: kbar_0 > kbar; closed economy vs selling all capital (e) vs optimal open economy (g).
kb = kbar()
p = policy(g=0.0)


def closed(k0):
    k, c = solve_path(k0, p)
    return k, c, welfare(c)


kb0 = 2 * kb
k, c, Wc = closed(kb0)
Rb = prices(k, c, p)["Rb"]
assert np.all(np.diff(k[:40]) < 0) and np.all(np.diff(c[:40]) < 0) and np.all(Rb[:40] < 1 + RHO)
c_sell = (1 - BE) * kb0                          # (e) as printed: c + beta A' = A, A_0 = kbar_0
W_sell = u(c_sell, GA) / (1 - BE)
c_opt = f(kb) - DE * kb + RHO * (kb0 - kb)       # (g) keep k = kbar, lend kbar_0 - kbar (timing of 11.6)
W_opt = u(c_opt, GA) / (1 - BE)
assert W_opt > Wc > W_sell and c_opt > RHO * kb0 > c_sell
print(f"kbar = {kb:.6f}, kbar_0 = {kb0:.6f}: closed c_0 = {c[0]:.6f}, Rbar_1 = {Rb[0]:.6f}; "
      f"sell-all c = (1-beta) kbar_0 = {c_sell:.6f} (rho kbar_0 = {RHO * kb0:.6f} with 11.6 timing); optimal open c = {c_opt:.6f}")
print(f"welfare: sell-all {W_sell:.4f} < closed {Wc:.4f} < optimal open {W_opt:.4f}")
m_star = brentq(lambda m: closed(m * kb)[2] - u((1 - BE) * m * kb, GA) / (1 - BE), 2.0, 200.0)
print(f"(f) selling everything beats autarky only if kbar_0 > {m_star:.3f} kbar = {m_star * kb:.3f}")
# (g)(ii): c(k) = f(k) - (rho + delta) k + rho kbar_0 is maximised at kbar, and c(kbar) > c(0) since f(kbar) > f'(kbar) kbar
kk = np.linspace(1e-6, 3 * kb, 30001)
assert abs(kk[np.argmax(f(kk) - (RHO + DE) * kk)] - kb) < 3 * kb / 30000 and f(kb) - (RHO + DE) * kb > 0

kbar = 1.489956, kbar_0 = 2.979913: closed c_0 = 1.116898, Rbar_1 = 0.969594; sell-all c = (1-beta) kbar_0 = 0.148996 (rho kbar_0 = 0.156838 with 11.6 timing); optimal open c = 0.921064
welfare: sell-all -134.2321 < closed -22.1754 < optimal open -21.7140
(f) selling everything beats autarky only if kbar_0 > 15.505 kbar = 23.101


## Exercise 11.9 --- Linear utility, finite horizon

A linear program; `linprog` vs the corner solutions.

In [10]:
# Exercise 11.9: linear utility, finite horizon: a linear program. T = 5, R = 1.05, y drawn once.
T9, R9 = 5, 1.05
y9 = np.random.default_rng(0).uniform(0.5, 1.5, T9 + 1)
disc = R9**-np.arange(T9 + 1)
W9 = disc @ y9                                   # present value of income
for b9, case in ((0.90, "beta R < 1"), (1 / R9 + 0.02, "beta R > 1"), (1 / R9, "beta R = 1")):
    obj = b9**np.arange(T9 + 1)
    lp = linprog(-obj, A_eq=disc[None, :], b_eq=[W9], bounds=[(0, None)] * (T9 + 1), method="highs")
    if b9 * R9 < 1:
        closed = np.r_[W9, np.zeros(T9)]
    elif b9 * R9 > 1:
        closed = np.r_[np.zeros(T9), W9 * R9**T9]
    else:
        closed = y9                              # one of many optima
    assert lp.status == 0 and np.isclose(obj @ closed, -lp.fun)
    if b9 * R9 != 1:
        assert np.allclose(lp.x, closed)
    print(f"{case}: LP c = {np.round(lp.x, 4)}, value {-lp.fun:.6f}; closed form value {obj @ closed:.6f}")
# with beta R = 1 the objective is sum R^-t c_t = W9 for every feasible path, e.g. a perturbed one
c_alt = y9 + 0.1 * np.r_[1, -R9, 0, 0, 0, 0]
assert np.isclose((1 / R9)**np.arange(T9 + 1) @ c_alt, W9)

beta R < 1: LP c = [4.9943 0.     0.     0.     0.     0.    ], value 4.994328; closed form value 4.994328
beta R > 1: LP c = [0.     0.     0.     0.     0.     6.3742], value 5.541225; closed form value 5.541225
beta R = 1: LP c = [4.9943 0.     0.     0.     0.     0.    ], value 4.994328; closed form value 4.994328


## Exercise 11.10 --- Term structure of interest rates

$r_{t-1,t}=\log(q_{t-1}/q_t)$ and $r_{0,t}$ (11.3.2). Check: $r_{t,t+1}=\log\bar R_{t+1}$ when $\tau_c$ is constant.

In [11]:
# Exercise 11.10: term structure at the steady state (a-c) and after an unforeseen permanent tau_k = .2 (d).
kb = kbar()
p0 = policy()
k, c = solve_path(kb, p0)
pr0 = prices(k, c, p0)
assert np.allclose(pr0["q"][:50], BE**np.arange(50)) and np.allclose(pr0["r"], -np.log(BE))
p10 = policy(tk=0.2)
k10, c10 = solve_path(kb, p10)
pr10 = prices(k10, c10, p10)
r_short = pr10["r"]                                     # r_{t-1,t}, t = 1, 2, ...
r_long = np.cumsum(r_short) / np.arange(1, len(r_short) + 1)   # r_{0,t} (11.3.2)
assert np.allclose(r_short, np.log(pr10["Rb"]))         # r_{t,t+1} = log Rbar_{t+1} when tau_c is constant
assert np.all(np.diff(r_short[:40]) > 0) and np.all(r_long[1:40] < r_short[1:40]) and np.all(r_short < -np.log(BE) + 1e-12)
print(f"(a)-(c) q_t = beta^t, r_(t-1,t) = r_(0,t) = -log beta = {-np.log(BE):.6f}")
print("(d) t:      " + " ".join(f"{s:7d}" for s in range(1, 11)))
print("  r_(t-1,t) " + " ".join(f"{x:.5f}" for x in r_short[:10]))
print("  r_(0,t)   " + " ".join(f"{x:.5f}" for x in r_long[:10]))
fig, ax = plt.subplots(1, 2, figsize=(7.5, 2.5))
t = np.arange(1, 11)
for a_, y, ti in ((ax[0], r_short, "$r_{t-1,t}$"), (ax[1], r_long, "$r_{0,t}$")):
    a_.plot(t, y[:10], "ko-", ms=3, label=r"$\tau_k=.2$ at $t=0$")
    a_.plot(t, 0 * t - np.log(BE), "k--", label="steady state (a)-(c)")
    a_.set_title(ti); a_.set_xlabel("t")
ax[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG + "ls11_10_yields.pdf"); plt.close(fig)

(a)-(c) q_t = beta^t, r_(t-1,t) = r_(0,t) = -log beta = 0.051293
(d) t:            1       2       3       4       5       6       7       8       9      10
  r_(t-1,t) 0.04255 0.04370 0.04469 0.04556 0.04632 0.04698 0.04755 0.04805 0.04848 0.04886
  r_(0,t)   0.04255 0.04312 0.04365 0.04413 0.04457 0.04497 0.04534 0.04568 0.04599 0.04627


## Exercise 11.11 --- A rising, concave consumption path

Two stories that produce figure 11.1; checks that $c$ rises at a falling rate and $\bar R_{t}>1+\rho$ falls.

In [12]:
# Exercise 11.11: stories for a rising, concave consumption path.
stories = {}
p = policy(tk=0.0)                                        # (i) unforeseen cut tau_k .2 -> 0 at t = 0 from its steady state
stories["cut in tau_k"] = (kbar(0.2), p)
stories["k_0 = .5 kbar"] = (0.5 * kbar(), policy())       # (ii) constant policy, capital below its steady state
fig, ax = plt.subplots(1, 3, figsize=(9, 2.4))
t = np.arange(41)
for (name, (k0, p)), st in zip(stories.items(), ("k-", "k--")):
    k, c = solve_path(k0, p)
    Rb = prices(k, c, p)["Rb"]
    dc = np.diff(c[:40])
    assert np.all(dc > 0) and np.all(np.diff(dc) < 0) and np.all(Rb[:40] > 1 + RHO) and np.all(np.diff(Rb[:40]) < 0)
    print(f"{name}: c_0 = {c[0]:.4f} -> {c[-1]:.4f}; k_0 = {k[0]:.4f} -> {k[-1]:.4f}; Rbar_1 = {Rb[0]:.4f} -> {1 + RHO:.4f}")
    ax[0].plot(t, c[:41], st, label=name); ax[1].plot(t, k[:41], st); ax[2].plot(t[1:], Rb[:40], st)
for a_, ti in zip(ax, ("$c_t$", "$k_t$", r"$\bar R_t$")):
    a_.set_title(ti)
ax[2].axhline(1 + RHO, color="gray", lw=.6, ls=":")
ax[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG + "ls11_11_rising_c.pdf"); plt.close(fig)

cut in tau_k: c_0 = 0.6204 -> 0.6426; k_0 = 1.3812 -> 1.4900; Rbar_1 = 1.0638 -> 1.0526
k_0 = .5 kbar: c_0 = 0.4665 -> 0.6426; k_0 = 0.7450 -> 1.4900; Rbar_1 = 1.1718 -> 1.0526


## Exercise 11.12 --- Flat, then a drop at $t=12$, then a decline

Story A: news at $t=12$ of $g=.2\to.4$ from $t=22$. Story B: at $t=12$ a surprise permanent rise in $g$ and $\tau_k$.

In [13]:
# Exercise 11.12: flat c until t = 12, a drop at 12, then a gradual decline. Two consistent stories, both news at t = 12.
kb, n12 = kbar(), 60
paths = {}
# A: at t = 12 the government announces g = .2 -> .4 from t = 22, lump-sum financed
pA = policy(g=jump(GB, 0.4, 10))
kA, cA = solve_path(kb, pA)
# B: at t = 12, unforeseen and immediate: g = .2 -> .4 and tau_k = 0 -> .2 (lump sums balance the rest)
pB = policy(g=0.4, tk=0.2)
kB, cB = solve_path(kb, pB)
c_old = f(kb) - DE * kb - GB
for name, k, c, p in (("A", kA, cA, pA), ("B", kB, cB, pB)):
    Rb = prices(k, c, p)["Rb"]
    kk = np.r_[np.full(12, kb), k[:n12 - 11]]
    cc = np.r_[np.full(12, c_old), c[:n12 - 12]]
    RR = np.r_[np.full(12, 1 + RHO), Rb[:n12 - 12]]
    gg = np.r_[np.full(12, GB), p["g"][:n12 - 12]]
    assert cc[12] < c_old and np.all(np.diff(cc[12:52]) < 0)          # the shape of figure 11.2
    paths[name] = kk, cc, RR, gg
    print(f"story {name}: c_11 = {cc[11]:.4f}, c_12 = {cc[12]:.4f}, c_40 = {cc[40]:.4f}; max k_t (t >= 12) = "
          f"{kk[12:40].max():.4f}, k_40 = {kk[40]:.4f}; Rbar_13 = {RR[12]:.4f}")
fig, ax = plt.subplots(1, 4, figsize=(11, 2.4))
t = np.arange(n12)
for (name, (kk, cc, RR, gg)), st in zip(paths.items(), ("k-", "k--")):
    lab = "A: news of g at 22" if name == "A" else r"B: surprise g and $\tau_k$ at 12"
    ax[0].plot(t, cc, st, label=lab); ax[1].plot(t, kk[:n12], st); ax[2].plot(t[1:], RR[:n12 - 1], st); ax[3].plot(t, gg, st)
for a_, ti in zip(ax, ("$c_t$", "$k_t$", r"$\bar R_t$", "$g_t$")):
    a_.set_title(ti); a_.axvline(12, color="gray", lw=.5, ls=":")
ax[0].legend(fontsize=6)
fig.tight_layout(); fig.savefig(FIG + "ls11_12_stories.pdf"); plt.close(fig)

story A: c_11 = 0.6426, c_12 = 0.6092, c_40 = 0.4518; max k_t (t >= 12) = 2.0985, k_40 = 1.5432; Rbar_13 = 1.0489
story B: c_11 = 0.6426, c_12 = 0.4548, c_40 = 0.4370; max k_t (t >= 12) = 1.4900, k_40 = 1.3854; Rbar_13 = 1.0432


## Exercise 11.13 --- Linear utility, infinite horizon

Closed form $k_{t+1}=\min\{f(k_t)+(1-\delta)k_t,\bar k\}$ vs grid value-function iteration; saving rate $s(k)$.

In [14]:
# Exercise 11.13: linear utility; bang-bang approach to kbar. Closed form vs value-function iteration.
kb = kbar()                                           # f'(kbar) = rho + delta
Y = lambda k: f(k) + (1 - DE) * k                     # resources available at t


def policy13(k):
    """k_{t+1} = min(Y(k), kbar); c = Y(k) - k_{t+1}."""
    return np.minimum(Y(k), kb)


grid = np.unique(np.r_[np.linspace(0.02, 3 * kb, 1500), kb])
Yg = Y(grid)
C = Yg[:, None] - grid[None, :]
Ucand = np.where(C >= 0, C, -np.inf)
V = np.zeros(len(grid))
for it in range(3000):
    Vn = np.max(Ucand + BE * V[None, :], axis=1)
    if np.max(np.abs(Vn - V)) < 1e-12:
        break
    V = Vn
pol = grid[np.argmax(Ucand + BE * V[None, :], axis=1)]
reach = Yg >= kb
assert np.allclose(pol[reach], kb)                                # jump to kbar whenever feasible
assert np.all(Yg[~reach] - pol[~reach] < np.max(np.diff(grid)) + 1e-12)   # otherwise save everything (c = 0 up to grid)
# closed-form value for reachable k: V(k) = Y(k) - kbar + beta cbar/(1-beta), cbar = f(kbar) - delta kbar
cbar13 = f(kb) - DE * kb
assert np.allclose(V[reach], Yg[reach] - kb + BE * cbar13 / (1 - BE), atol=1e-8)
k_low = brentq(lambda k: Y(k) - kb, 1e-6, kb)                     # reachable in one step iff k >= k_low
s13 = lambda k: np.minimum(1.0, (kb - (1 - DE) * k) / f(k))       # saving rate (d)
kk = np.linspace(0.05, 4 * kb, 400)
assert np.all(np.diff(s13(kk[kk > k_low])) < 0) and np.isclose(s13(kb), DE * AL / (RHO + DE))
print(f"kbar = {kb:.6f}, cbar = {cbar13:.6f}; one-step reach threshold k = {k_low:.6f}; VFI converged in {it} iterations")
print(f"saving rate at kbar = delta alpha/(rho+delta) = {s13(kb):.6f}; s(k) < 0 for k > kbar/(1-delta) = {kb / (1 - DE):.6f}")
kpath, cpath = [0.05], []
for _ in range(12):
    kn = policy13(kpath[-1]); cpath.append(Y(kpath[-1]) - kn); kpath.append(kn)
print("from k_0 = .05: k =", np.round(kpath[:8], 4), " c =", np.round(cpath[:8], 4))
fig, ax = plt.subplots(1, 2, figsize=(7.5, 2.5))
ax[0].plot(grid, pol, "k-", grid, grid, "k:", lw=1); ax[0].set_title("policy $k_{t+1}(k_t)$ (VFI)"); ax[0].set_xlabel("$k_t$")
ax[1].plot(kk, s13(kk), "k-"); ax[1].axhline(0, color="gray", lw=.5); ax[1].axvline(kb, color="gray", lw=.5, ls="--")
ax[1].set_title("saving rate $s(k_t)$"); ax[1].set_xlabel("$k_t$")
fig.tight_layout(); fig.savefig(FIG + "ls11_13_linear.pdf"); plt.close(fig)

kbar = 1.489956, cbar = 0.842645; one-step reach threshold k = 0.733812; VFI converged in 534 iterations
saving rate at kbar = delta alpha/(rho+delta) = 0.261250; s(k) < 0 for k > kbar/(1-delta) = 1.862446
from k_0 = .05: k = [0.05   0.4121 1.076  1.49   1.49   1.49   1.49   1.49  ]  c = [0.     0.     0.3954 0.8426 0.8426 0.8426 0.8426 0.8426]


## Exercise 11.14 --- Transition with and without government purchases

$k_0=.5\bar k$; $\phi=.1$ so that $g=\phi f(k_0)$ can also be financed by a capital tax in (e).

In [15]:
# Exercise 11.14: k_0 = .5 kbar; g = 0 (c), g = phi f(k_0) lump-sum (d), g financed by a constant tau_k (e).
kb = kbar()
k0 = 0.5 * kb
s_ss = DE * kb / f(kb)
assert np.isclose(s_ss, DE * AL / (RHO + DE))
PHI = 0.1
g14 = PHI * f(k0)
res14 = {}
for name, p in (("c: g = 0", policy(g=0.0)), ("d: g lump sum", policy(g=g14))):
    k, c = solve_path(k0, p)
    ks, cs = shoot(k0, p)
    assert np.allclose(ks[:40], k[:40], atol=1e-8)
    res14[name] = k, c, prices(k, c, p)["Rb"]
(kc, cc, Rc), (kd, cd, Rd) = res14.values()
assert np.isclose(kc[-1], kd[-1]) and np.allclose(Rc[-5:], Rd[-5:])       # same kbar and Rbar in the limit
assert np.all(kd[1:40] < kc[1:40]) and np.all(Rd[:40] > Rc[:40])          # g slows capital accumulation
assert np.isclose(cc[-1] - cd[-1], g14)


def rev14(tk):
    """PV of capital-tax revenue minus PV of g for a constant tau_k from t = 0 (no lump sums)."""
    p = policy(g=g14, tk=tk)
    k, c = solve_path(k0, p)
    q = prices(k, c, p)["q"]
    return pv(tk * (fp(k[:len(c)]) - DE) * k[:len(c)], q) - pv(g14 + 0 * c, q)


tk14 = brentq(rev14, 0.01, 0.85)
pe = policy(g=g14, tk=tk14)
ke, ce = solve_path(k0, pe)
Re = prices(ke, ce, pe)["Rb"]
print(f"(b) kbar = {kb:.6f}, saving rate x/f = delta alpha/(rho+delta) = {s_ss:.6f}")
print(f"(c) k_0 = {k0:.6f}: c_0 = {cc[0]:.6f}, Rbar_1 = {Rc[0]:.6f}; (d) g = {PHI} f(k_0) = {g14:.6f}: c_0 = {cd[0]:.6f}, "
      f"Rbar_1 = {Rd[0]:.6f}; k_10 = {kc[10]:.6f} vs {kd[10]:.6f}")
print(f"(e) revenue-neutral constant tau_k = {tk14:.6f}; steady state k = {kbar(tk14):.6f}, c_0 = {ce[0]:.6f}, k_10 = {ke[10]:.6f}")
fig, ax = plt.subplots(1, 3, figsize=(9, 2.4))
t = np.arange(41)
for (k, c, Rb), st, lab in (((kc, cc, Rc), "k-", "g = 0"), ((kd, cd, Rd), "k--", "g lump sum"), ((ke, ce, Re), "k:", r"g by $\tau_k$")):
    ax[0].plot(t, c[:41], st, label=lab); ax[1].plot(t, k[:41], st); ax[2].plot(t[1:], Rb[:40], st)
for a_, ti in zip(ax, ("$c_t$", "$k_{t}$", r"$\bar R_{t}$")):
    a_.set_title(ti)
ax[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG + "ls11_14_paths.pdf"); plt.close(fig)

(b) kbar = 1.489956, saving rate x/f = delta alpha/(rho+delta) = 0.261250
(c) k_0 = 0.744978: c_0 = 0.647540, Rbar_1 = 1.166272; (d) g = 0.1 f(k_0) = 0.090742: c_0 = 0.565017, Rbar_1 = 1.168648; k_10 = 1.360230 vs 1.342656
(e) revenue-neutral constant tau_k = 0.641199; steady state k = 0.929016, c_0 = 0.649316, k_10 = 0.866239


## Exercise 11.15 --- A rising, concave yield curve

Two stories producing figure 11.3: a surprise permanent $\tau_k$ at $t=0$, or $k_0>\bar k$.

In [16]:
# Exercise 11.15: a rising, concave yield curve r_(0,t) (figure 11.3); two consistent stories.
kb = kbar()
fig, ax = plt.subplots(1, 3, figsize=(9, 2.4))
t = np.arange(1, 41)
for (name, k0, p), st in ((("surprise tau_k = .2 at 0", kb, policy(tk=0.2)), "k-"),
                          (("k_0 = 1.5 kbar", 1.5 * kb, policy()), "k--")):
    k, c = solve_path(k0, p)
    r = prices(k, c, p)["r"]
    y = np.cumsum(r) / np.arange(1, len(r) + 1)
    assert np.all(np.diff(y[:40]) > 0) and np.all(np.diff(y[:40], 2) < 0)           # rising and concave
    assert np.all(np.diff(c[:40]) < 0) and np.all(np.diff(k[:40]) < 0)              # c and k fall
    print(f"{name}: r_(0,1) = {y[0]:.5f}, r_(0,10) = {y[9]:.5f}, r_(0,40) = {y[39]:.5f} -> {-np.log(BE):.5f}; "
          f"c_0 = {c[0]:.4f} -> {c[-1]:.4f}")
    ax[0].plot(t, y[:40], st, label=name); ax[1].plot(t - 1, c[:40], st); ax[2].plot(t - 1, k[:40], st)
for a_, ti in zip(ax, ("$r_{0,t}$", "$c_t$", "$k_t$")):
    a_.set_title(ti)
ax[0].axhline(-np.log(BE), color="gray", lw=.6, ls=":"); ax[0].legend(fontsize=6)
fig.tight_layout(); fig.savefig(FIG + "ls11_15_yield_curve.pdf"); plt.close(fig)

surprise tau_k = .2 at 0: r_(0,1) = 0.04255, r_(0,10) = 0.04627, r_(0,40) = 0.04965 -> 0.05129; c_0 = 0.6575 -> 0.6362
k_0 = 1.5 kbar: r_(0,1) = -0.00024, r_(0,10) = 0.02171, r_(0,40) = 0.04185 -> 0.05129; c_0 = 0.7765 -> 0.6426


## Exercise 11.16 --- High vs low $\gamma$

$\gamma=.2$ vs $\gamma=5$ from $k_0=.5\bar k$; Newton vs shooting; $\lambda_2(\gamma)$.

In [17]:
# Exercise 11.16: same k_0 = .5 kbar and constant policy (g = .2), gamma = .2 vs gamma = 5.
kb = kbar()
p = policy()
res16 = {}
for gam in (0.2, 5.0):
    k, c = solve_path(0.5 * kb, p, gam=gam)
    ks, cs = shoot(0.5 * kb, p, gam=gam)
    assert np.allclose(ks[:30], k[:30], atol=1e-7)
    res16[gam] = k, c
(kl, cl), (kh, ch) = res16[0.2], res16[5.0]
assert np.all(kl[1:30] > kh[1:30]) and cl[0] < ch[0] and np.any(cl[:30] > ch[:30])     # fast vs slow; c paths cross
cross = np.argmax(cl > ch)
print(f"low gamma: k_5 = {kl[5]:.4f}, c_0 = {cl[0]:.4f}, lambda_2 = {lam2_root(0.2):.4f}; "
      f"high gamma: k_5 = {kh[5]:.4f}, c_0 = {ch[0]:.4f}, lambda_2 = {lam2_root(5.0):.4f}; c paths cross at t = {cross}")
fig, ax = plt.subplots(1, 3, figsize=(9, 2.4))
t = np.arange(41)
for gam, st in ((0.2, "k-"), (5.0, "k--")):
    k, c = res16[gam]
    ax[0].plot(t, k[:41], st, label=rf"$\gamma={gam}$"); ax[1].plot(t, fp(k[:41]), st); ax[2].plot(t, c[:41], st)
for a_, ti, y in zip(ax, ("$k_t$", "$f'(k_t)$", "$c_t$"), (kb, RHO + DE, f(kb) - DE * kb - GB)):
    a_.set_title(ti); a_.axhline(y, color="gray", lw=.6, ls=":")
ax[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG + "ls11_16_gamma.pdf"); plt.close(fig)

low gamma: k_5 = 1.4287, c_0 = 0.2945, lambda_2 = 0.5779; high gamma: k_5 = 0.9780, c_0 = 0.5071, lambda_2 = 0.9109; c paths cross at t = 3


## Exercise 11.17 --- Anticipated vs unanticipated jump in $g$

$g=.2\to.4$ at $t=10$.

In [18]:
# Exercise 11.17: g = .2 -> .4 at t = 10, anticipated since 0 vs a surprise at 10.
kb = kbar()
p = policy(g=jump(GB, 0.4, 10))
ka, ca = solve_path(kb, p)
Ra = prices(ka, ca, p)["Rb"]
c_old, c_new = f(kb) - DE * kb - GB, f(kb) - DE * kb - 0.4
# surprise: before 10 the old steady state; at 10 the economy is at kbar with the new g: k constant, c drops by .2
ku, cu = solve_path(kb, policy(g=0.4))
assert np.allclose(ku, kb) and np.allclose(cu, c_new)
cs_ = np.r_[np.full(10, c_old), cu[:31]]
assert np.all(np.diff(ca[:40]) < 0) and ka[10] == ka.max() and Ra[9] == Ra.min() and np.all(Ra[:40] < 1 + RHO)
print(f"anticipated: c_0 = {ca[0]:.4f}, c_10 = {ca[10]:.4f}, c_40 = {ca[40]:.4f}; k_10 = {ka[10]:.4f}; min Rbar = Rbar_10 = {Ra[9]:.4f}")
print(f"surprise: c = {c_old:.4f} for t < 10, {c_new:.4f} for t >= 10; k = {kb:.4f} and Rbar = {1 + RHO:.4f} throughout")
fig, ax = plt.subplots(1, 4, figsize=(11, 2.4))
t = np.arange(41)
ax[0].plot(t, cs_, "k-"); ax[0].set_title("(a) $c_t$: surprise at 10")
ax[1].plot(t, ca[:41], "k-"); ax[1].set_title("(b) $c_t$: anticipated since 0")
ax[2].plot(t, ka[:41], "k-", t, 0 * t + kb, "k--"); ax[2].set_title("$k_t$ (dashed: surprise)")
ax[3].plot(t[1:], Ra[:40], "k-", t, 0 * t + 1 + RHO, "k--"); ax[3].set_title(r"$\bar R_t$ (dashed: surprise)")
fig.tight_layout(); fig.savefig(FIG + "ls11_17_news.pdf"); plt.close(fig)

anticipated: c_0 = 0.6092, c_10 = 0.5390, c_40 = 0.4446; k_10 = 2.0985; min Rbar = Rbar_10 = 1.0008
surprise: c = 0.6426 for t < 10, 0.4426 for t >= 10; k = 1.4900 and Rbar = 1.0526 throughout


## Exercise 11.18 --- Habits and durability

The book's $\alpha$ in $u(c_t-\alpha c_{t-1})$ is written `a` (to keep $\alpha=.33$ for $f$), $g=0$. Newton on the Euler
equations for $k_1,\dots,k_n$; check against the stable subspace of the linearised 4-dimensional system
$(k_t,c_{t-1},c_t,c_{t+1})$ and against the standard model at $a=0$.

In [19]:
# Exercise 11.18: habit (a > 0) or durability (a < 0): u(c_t - a c_{t-1}); a is the book's alpha in the utility.
kb = kbar()                                   # (c): f'(k) = rho + delta, as without habits
cb18 = f(kb) - DE * kb
Y = lambda k: f(k) + (1 - DE) * k
up18 = lambda x: x**(-GA)


def lam18(cm, c, cp, a):
    """lambda_t = u'(c_t - a c_{t-1}) - a beta u'(c_{t+1} - a c_t): FOC for c_t (a)."""
    return up18(c - a * cm) - a * BE * up18(cp - a * c)


def habit_path(k0, cm1, a, n=N):
    """Newton on the Euler equations lambda_t = beta lambda_{t+1} (f'(k_{t+1}) + 1 - delta), t = 0..n-1,
    unknowns k_1..k_n, with k_{n+1} = k_{n+2} = kbar. Returns k_0..k_{n+1}, c_{-1}..c_{n+1}."""
    def unpack(x):
        k = np.r_[k0, x, kb, kb]
        return k, np.r_[cm1, Y(k[:-1]) - k[1:]]           # c_{-1}, c_0..c_{n+1}

    def resid(x):
        k, c = unpack(x)
        lam = lam18(c[:-2], c[1:-1], c[2:], a)            # lambda_0..lambda_n
        return lam[:-1] / (BE * lam[1:] * (fp(k[1:n + 1]) + 1 - DE)) - 1

    x = newton(resid, kb + (k0 - kb) * 0.8**np.arange(1, n + 1), ncol=5)
    return unpack(x)


# steady state (b)-(c): lambda = (1 - a beta) u'((1-a) c) > 0 and beta (f'(k) + 1 - delta) = 1
for a in (0.5, -0.3):
    lam_ss = lam18(cb18, cb18, cb18, a)
    assert np.isclose(lam_ss, (1 - a * BE) * up18((1 - a) * cb18)) and lam_ss > 0
# a = 0 reproduces the standard model
k0, cm1 = 0.5 * kb, f(0.5 * kb) - DE * 0.5 * kb
k_a0, c_a0 = habit_path(k0, cm1, 0.0)
k_std, c_std = solve_path(k0, policy(g=0.0))
assert np.allclose(k_a0[:60], k_std[:60], atol=1e-8)


def next_state(s, a):
    """(k_t, c_{t-1}, c_t, c_{t+1}) -> (k_{t+1}, c_t, c_{t+1}, c_{t+2}) from feasibility and the Euler equation."""
    k, cm, c, cp = s
    kn = Y(k) - c
    lam = lam18(cm, c, cp, a)
    m = (up18(cp - a * c) - lam / (BE * (fp(kn) + 1 - DE))) / (a * BE)
    return np.array([kn, c, cp, a * cp + m**(-1 / GA)])


res18 = {}
for a in (0.5, -0.3):
    sb = np.array([kb, cb18, cb18, cb18])
    J = np.column_stack([(next_state(sb + h, a) - next_state(sb - h, a)) / 2e-7 for h in 1e-7 * np.eye(4)])
    ev, V = np.linalg.eig(J)
    st = np.abs(ev) < 1
    assert st.sum() == 2                                # two predetermined (k_t, c_{t-1}), two jump variables
    # roots come in pairs (lambda, 1/(beta lambda)), as in section 11.10.1
    assert np.allclose(np.sort(1 / (BE * ev[st].real)), np.sort(ev[~st].real), rtol=1e-5)
    Vs = V[:, st]
    dk0 = -0.01 * kb                                    # small displacement: k_0 = .99 kbar, c_{-1} = cbar
    w = np.linalg.solve(Vs[:2], [dk0, 0.0])
    # iterate in eigen-coordinates (iterating J itself would pick up round-off along the unstable roots)
    k_lin = kb + np.real(np.array([Vs[0] @ (ev[st]**t * w) for t in range(61)]))
    k_nl, c_nl = habit_path(kb + dk0, cb18, a)
    assert np.max(np.abs(k_lin - k_nl[:61])) < 0.01 * abs(dk0)
    res18[a] = habit_path(k0, cm1, a)
    print(f"a = {a}: eigenvalues {np.round(np.sort(ev.real), 5)}; lambda_ss = {lam18(cb18, cb18, cb18, a):.5f}; "
          f"linear vs Newton max |dk| = {np.max(np.abs(k_lin - k_nl[:61])):.2e} (deviation {abs(dk0):.4f})")
res18[0.0] = k_a0, c_a0
print(f"kbar = {kb:.6f}, f'(kbar) = {fp(kb):.6f}, cbar = {cb18:.6f} (independent of a)")
fig, ax = plt.subplots(1, 2, figsize=(7.5, 2.5))
t = np.arange(41)
for a, stl in ((0.0, "k-"), (0.5, "k--"), (-0.3, "k:")):
    k, c = res18[a]
    ax[0].plot(t, k[:41], stl, label=f"a = {a}"); ax[1].plot(t, c[1:42], stl)
ax[0].set_title("$k_t$"); ax[1].set_title("$c_t$"); ax[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG + "ls11_18_habit.pdf"); plt.close(fig)

a = 0.5: eigenvalues [0.51624 0.82191 1.28072 2.03903]; lambda_ss = 2.95753; linear vs Newton max |dk| = 1.23e-05 (deviation 0.0149)
a = -0.3: eigenvalues [-3.55805 -0.29585  0.83091  1.26685]; lambda_ss = 1.07085; linear vs Newton max |dk| = 6.40e-06 (deviation 0.0149)
kbar = 1.489956, f'(kbar) = 0.252632, cbar = 0.842645 (independent of a)


## Exercise 11.19 --- The invisible hand, I

Big $K$: the aggregate path for the foreseen jump in $g$ (section 11.9). Little $k$: the household's problem at those
prices, solved in closed form ($c_t=(\mu q_t/\beta^t)^{-1/\gamma}$, $\mu$ from the budget). Check: $c=C$, $k=K$.

In [20]:
# Exercise 11.19: Big K, little k. Aggregate path (big K) for the foreseen jump in g at t = 10 by Newton;
# then the household's own problem at those prices, solved independently in closed form.
kb = kbar()
p = policy(g=jump(GB, 0.4, 10))
K, C = solve_path(kb, p)                      # big K: aggregate law of motion (11.6.2)
pr = prices(K, C, p)
q, eta, w = pr["q"], pr["eta"], pr["w"]
n = len(q)
tau_h = p["g"][:n]                            # lump-sum taxes: balanced budget date by date (any PV-equivalent timing works)
W0 = (eta[0] + 1 - DE) * K[0] * q[0] + pv(w - tau_h, q)          # household wealth at the given prices
# u'(c_t) beta^t = mu q_t  =>  c_t = (mu q_t / beta^t)^(-1/gamma); mu from the budget sum q_t c_t = W0
shape = (q / BE**np.arange(n))**(-1 / GA)
mu = (pv(shape, q) / W0)**GA
c_hh = mu**(-1 / GA) * shape
assert np.allclose(c_hh, C, rtol=1e-6)                          # little c = big C
k_hh = [K[0]]
for t in range(60):
    k_hh.append((eta[t] + 1 - DE) * k_hh[-1] + w[t] - tau_h[t] - c_hh[t])
assert np.allclose(k_hh, K[:61], atol=1e-5)                      # little k = big K (with tau_h = g_t)
assert np.allclose(q[:-1] / q[1:], eta[1:] + 1 - DE)             # no-arbitrage (11.5.2): household indifferent about k
print(f"mu = {mu:.6f}; household c_0 = {c_hh[0]:.6f} = C_0 = {C[0]:.6f}; max |k - K| over 60 periods = "
      f"{np.max(np.abs(np.array(k_hh) - K[:61])):.2e}")

mu = 2.694141; household c_0 = 0.609242 = C_0 = 0.609242; max |k - K| over 60 periods = 2.78e-10


## Exercise 11.20 --- The invisible hand, II

Section 11.12 economy, $B=3$, $g=.2$, $\tau_n=0\to.2$ at $t=10$ (figure 11.12.3). Newton on (11.12.15) vs shooting on
$c_0$.

In [21]:
# Exercise 11.20: elastic labour, U = ln c + B(1 - n), B = 3 (section 11.12); tau_n = 0 -> .2 at t = 10, foreseen.
B20, TN = 3.0, 0.2
F = lambda k, n: k**AL * n**(1 - AL)
Fk = lambda k, n: AL * (k / n)**(AL - 1)
Fn = lambda k, n: (1 - AL) * (k / n)**AL
kap = kbar()                                           # steady-state k/n from (11.12.12), tau_k = 0


def ss20(tn):
    """(k, c, n) from (11.12.12)-(11.12.14)."""
    c = (1 - tn) * (1 - AL) * kap**AL / B20
    n = (c + GB) / (f(kap) - DE * kap)
    return n * kap, c, n


def elastic_path(k0, tn, n=120):
    """Newton on (11.12.15): unknowns k_1..k_n, n_0..n_n; k_{n+1} = terminal steady state."""
    kT = ss20(tn[-1])[0]

    def unpack(x):
        k = np.r_[k0, x[:n], kT]
        h = x[n:]
        return k, h, F(k[:-1], h) + (1 - DE) * k[:-1] - GB - k[1:]

    def resid(x):
        k, h, c = unpack(x)
        Rb = 1 + Fk(k[1:n + 1], h[1:]) - DE
        return np.r_[c[1:] / c[:-1] - BE * Rb, B20 * c - (1 - tn[:n + 1]) * Fn(k[:-1], h)]

    k_, c_, h_ = ss20(0.0)
    x = newton(resid, np.r_[np.full(n, k_), np.full(n + 1, h_)])
    return unpack(x)


def elastic_shoot(k0, tn, n=60):
    """Shooting on c_0: n_t from B c_t = (1-tau_n) F_n (kappa_t = k_t/n_t), c_{t+1} = beta Rbar_{t+1} c_t with
    kappa_{t+1} solving (1-tau_n,t+1)(1-alpha) kappa^alpha / B = beta [1 + f'(kappa) - delta] c_t."""
    kT = ss20(tn[-1])[0]

    def run(c0):
        k, c, h = [k0], [c0], []
        for t in range(n):
            kp = (B20 * c[t] / ((1 - tn[t]) * (1 - AL)))**(1 / AL)
            h.append(k[t] / kp)
            k.append(F(k[t], h[t]) + (1 - DE) * k[t] - GB - c[t])
            if k[-1] <= 0:
                return -1, k, c, h
            if k[-1] > 20 * kT:
                return 1, k, c, h
            kn = brentq(lambda x: (1 - tn[t + 1]) * (1 - AL) * x**AL / B20 - BE * (1 + fp(x) - DE) * c[t], 1e-8, 1e4)
            c.append((1 - tn[t + 1]) * (1 - AL) * kn**AL / B20)
        return np.sign(k[n] - kT), k, c, h

    lo, hi = 1e-6, 2.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if run(mid)[0] > 0:
            lo = mid
        else:
            hi = mid
        if hi - lo < 1e-16:
            break
    return [np.array(v) for v in run(lo)[1:]]


tn = jump(0.0, TN, 10, n=122)
k0, c0_, n0_ = ss20(0.0)
k, h, c = elastic_path(k0, tn)
ks, cs, hs = elastic_shoot(k0, tn)
assert np.allclose(ks[:30], k[:30], atol=1e-7) and np.allclose(hs[:30], h[:30], atol=1e-7)
w20 = Fn(k[:-1], h)
Rb20 = 1 + Fk(k[1:-1], h[1:]) - DE
k1, c1, n1 = ss20(TN)
assert np.isclose(k[-2], k1, rtol=1e-6) and np.all(h[:10] > n0_) and np.all(np.diff(k[:11]) > 0)
assert np.max(np.abs(c[:10] / c0_ - 1)) < 2e-3                     # consumption (almost) unaltered before t = 10
assert np.allclose(B20 * c, (1 - tn[:len(c)]) * w20)               # c pinned by the after-tax wage (11.12.15c)
print(f"old steady state k = {k0:.4f}, c = {c0_:.4f}, n = {n0_:.4f}; new k = {k1:.4f}, c = {c1:.4f}, n = {n1:.4f}")
print(f"c_0..c_9 / c_old - 1 in [{np.min(c[:10] / c0_ - 1):.2e}, {np.max(c[:10] / c0_ - 1):.2e}]; n_9 = {h[9]:.4f}, "
      f"k_10 = {k[10]:.4f}, n_10 = {h[10]:.4f}, w_10 = {w20[10]:.4f}, Rbar_10 = {Rb20[9]:.4f}")
fig, ax = plt.subplots(2, 3, figsize=(9, 4.6))
t = np.arange(41)
for a_, y, y0, ti in ((ax[0, 0], k[:41], k0, "$k$"), (ax[0, 1], c[:41], c0_, "$c$"), (ax[0, 2], h[:41], n0_, "$n$"),
                      (ax[1, 0], Rb20[:40], 1 + RHO, r"$\bar R$"), (ax[1, 1], w20[:41], w20[0] * 0 + (1 - AL) * kap**AL, "$w$"),
                      (ax[1, 2], tn[:41], 0.0, r"$\tau_n$")):
    a_.plot(np.arange(len(y)) + (1 if ti == r"$\bar R$" else 0), y, "k-"); a_.axhline(y0, color="k", ls="--", lw=.7); a_.set_title(ti)
fig.tight_layout(); fig.savefig(FIG + "ls11_20_tau_n.pdf"); plt.close(fig)

old steady state k = 0.8041, c = 0.2547, n = 0.5397; new k = 0.7140, c = 0.2038, n = 0.4792
c_0..c_9 / c_old - 1 in [-1.15e-03, -3.24e-05]; n_9 = 0.6136, k_10 = 0.9764, n_10 = 0.4087, w_10 = 0.8931, Rbar_10 = 0.9841
